# Extraction des zones vers Heurist

In [109]:
import re, os, csv, io, json
import pandas as pd
import glob



# --- Config à ajouter ---
ZONES_INPUT_DIR = "../List-of-zones"
ZONES_GLOB_PATTERN = "*_labelstudio.csv"
IMAGES_CSV_PATH = "../List-of-images/JJ096-JJ211_image_data_with_resolved_urls.csv"  

ARK_ID_RE = re.compile(r"ark:/\d+/[A-Za-z0-9]+")



# --- Images à exclure explicitement (doublons confirmés) ---
EXCLUDED_ARK_IDS = [
    "ark:/63955/vhvhlrwi8q9b",  # JJ113, doublon
    "ark:/63955/vs7r351na0th",  # JJ113, doublon
]

In [110]:
# --- Table de correspondance des noms de colonnes alternatifs ---
COLUMN_ALIASES = {
    "registre": "register",
    "ordre": "folio_sort_key",
    # ajoutez ici d'autres variantes repérées (ex: "folio": "folio_label")
}


def normalize_zone_columns(df: pd.DataFrame, source_file: str) -> pd.DataFrame:
    """Renomme les colonnes selon COLUMN_ALIASES (comparaison insensible à la casse/espaces)."""
    rename_map = {}
    for col in df.columns:
        key = col.strip().lower()
        if key in COLUMN_ALIASES:
            rename_map[col] = COLUMN_ALIASES[key]
    if rename_map:
        print(f"  Normalisation colonnes ({source_file}): {rename_map}")
        df = df.rename(columns=rename_map)
    return df

def check_missing_canonical_columns(zones_df: pd.DataFrame,
                                     expected=("register", "folio_label", "folio_sort_key",
                                               "image", "image_path", "label")):
    """Signale, par fichier source, les colonnes canoniques absentes après normalisation."""
    for source_file, group in zones_df.groupby("__source_file"):
        missing = [c for c in expected if c not in zones_df.columns or group[c].isna().all()]
        if missing:
            print(f"⚠️  {source_file}: colonnes manquantes/vides après normalisation: {missing}")
            

def sniff_delimiter(sample_text: str) -> str:
    """Détecte le séparateur (tabulation, virgule, point-virgule...) à partir d'un échantillon."""
    try:
        dialect = csv.Sniffer().sniff(sample_text, delimiters="\t,;|")
        return dialect.delimiter
    except csv.Error:
        # Par défaut on suppose une tabulation (format observé dans les exemples fournis)
        return "\t"


def extract_ark_id(url: str):
    """Extrait l'identifiant ark:/xxxxx/yyyyy d'une URL IIIF, indépendamment du suffixe de taille."""
    if not isinstance(url, str):
        return None
    m = ARK_ID_RE.search(url)
    return m.group(0) if m else None


def extract_base_image_id(physical_url: str):
    """
    À partir d'une URL image physique IIIF (.../DEPOT/{base_image}/{region}/{size}/{rotation}/{quality}.{ext}),
    extrait l'identifiant {base_image} (ex. FRCHANJJ_JJ037_0004R_A).
    """
    if not isinstance(physical_url, str):
        return None
    parts = physical_url.rstrip("/").split("/")
    if len(parts) < 5:
        return None
    return parts[-5]


def read_zone_export(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8", errors="strict") as f:
        text = f.read()
    delimiter = sniff_delimiter(text[:5000])
    df = pd.read_csv(io.StringIO(text), sep=delimiter, dtype=str, keep_default_na=False, na_values=[""])
    df = normalize_zone_columns(df, os.path.basename(filepath))
    df["__source_file"] = os.path.basename(filepath)
    return df


def load_all_zone_exports(input_dir: str, pattern: str) -> pd.DataFrame:
    """Concatène tous les exports LS d'un dossier."""
    filepaths = sorted(glob.glob(os.path.join(input_dir, pattern)))
    if not filepaths:
        raise FileNotFoundError(f"Aucun export LS trouvé dans '{input_dir}' avec le motif '{pattern}'.")
    frames = []
    for fp in filepaths:
        print(f"Lecture export LS: {fp}")
        frames.append(read_zone_export(fp))
    combined = pd.concat(frames, ignore_index=True, sort=False)
    print(f"\nTotal: {len(frames)} fichiers, {len(combined)} lignes (images annotées).")
    return combined




def exclude_known_duplicate_images(zones_df: pd.DataFrame, excluded_ark_ids: list) -> pd.DataFrame:
    """Retire du corpus les images identifiées comme doublons (par ark_id), avant explosion des zones."""
    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)

    mask_excluded = df["ark_id"].isin(excluded_ark_ids)
    n_excluded = mask_excluded.sum()
    if n_excluded:
        print(f"🗑️  {n_excluded} ligne(s) exclue(s) (doublons confirmés) :")
        cols = [c for c in ["register", "ark_id", "image_path", "folio_sort_key", "__source_file"] if c in df.columns]
        print(df.loc[mask_excluded, cols].to_string(index=False))

    return df[~mask_excluded].drop(columns=["ark_id"]).reset_index(drop=True)


def read_images_csv(filepath: str) -> pd.DataFrame:
    """
    Lit le CSV des images déjà fusionné et résolu (sortie du premier notebook,
    colonnes urlImage_arkId / urlImage). Écrit en UTF-8, séparateur ',' par défaut
    (pandas.to_csv standard).
    """
    df = pd.read_csv(filepath, dtype=str, keep_default_na=False, na_values=[""])

    required_cols = {"urlImage_arkId", "urlImage"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Colonnes manquantes dans {filepath}: {missing}. "
            f"Est-ce bien le fichier images_merged_with_resolved_urls.csv (sortie du 1er notebook) ?"
        )

    print(f"{len(df)} lignes lues depuis {filepath}.")
    n_missing_physical = (df["urlImage"] == "").sum() if "" in df["urlImage"].values else df["urlImage"].isna().sum()
    if n_missing_physical:
        print(f"⚠️  {n_missing_physical} ligne(s) sans URL physique résolue (urlImage vide).")
    return df


def build_unique_images_from_zones(zones_df: pd.DataFrame, images_df: pd.DataFrame,
                                    include_unannotated: bool = True):
    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)

    dup_mask = df.duplicated(subset=["register", "ark_id"], keep=False)
    if dup_mask.any():
        n_dup = dup_mask.sum()
        print(f"⚠️  ATTENTION: {n_dup} lignes en doublon détectées sur (register, ark_id) !")
        print(df.loc[dup_mask, ["register", "ark_id", "image_path", "__source_file"]].to_string(index=False))
    else:
        print("Aucun doublon (register, ark_id) détecté.")

    df["n_zones"] = df["label"].apply(lambda x: len(json.loads(x)) if isinstance(x, str) and x.strip() else 0)
    if not include_unannotated:
        before = len(df)
        df = df[df["n_zones"] > 0].copy()
        print(f"Filtrage images sans zone: {before} -> {len(df)} lignes.")

    # --- Jointure élargie : on récupère aussi imageLabel comme repli pour folio_label ---
    images_lookup = images_df.copy()
    images_lookup["ark_id"] = images_lookup["urlImage_arkId"].apply(extract_ark_id)
    images_lookup = images_lookup.drop_duplicates(subset=["ark_id"])[["ark_id", "urlImage", "imageLabel"]]

    df = df.merge(images_lookup, on="ark_id", how="left", suffixes=("", "_from_images_list"))

    # --- Complète folio_label manquant/vide avec imageLabel (liste d'images) ---
    if "folio_label" not in df.columns:
        df["folio_label"] = pd.NA
    missing_folio = df["folio_label"].isna() | (df["folio_label"].astype(str).str.strip() == "")
    n_filled = (missing_folio & df["imageLabel"].notna() & (df["imageLabel"].astype(str).str.strip() != "")).sum()
    df.loc[missing_folio, "folio_label"] = df.loc[missing_folio, "imageLabel"]
    if n_filled:
        print(f"folio_label complété depuis imageLabel pour {n_filled} ligne(s).")
    still_missing = (df["folio_label"].isna() | (df["folio_label"].astype(str).str.strip() == "")).sum()
    still_missing_mask = df["folio_label"].isna() | (df["folio_label"].astype(str).str.strip() == "")
    still_missing = still_missing_mask.sum()
    if still_missing:
        print(f"⚠️  {still_missing} ligne(s) encore sans folio_label (ni zones ni images).")
        cols = [c for c in ["register", "ark_id", "image_path", "folio_sort_key", "__source_file"] if c in df.columns]
        print(df.loc[still_missing_mask, cols].to_string(index=False))
        
    # --- Suppression des images sans correspondance physique, puis reset de l'index ---
    n_before = len(df)
    unmatched_df = df[df["urlImage"].isna() | (df["urlImage"] == "")].copy()
    df = df[df["urlImage"].notna() & (df["urlImage"] != "")].reset_index(drop=True)
    n_removed = n_before - len(df)
    if n_removed:
        print(f"🗑️  {n_removed} image(s) sans correspondance supprimée(s) de la liste.")

    df["base_image"] = df["urlImage"].apply(extract_base_image_id)

    
    # --- ID temporaire, préfixé par registre ---
    df["temp_id1"] = (
        df.groupby("register").cumcount().add(1).astype(str).str.zfill(4)
    )
    df["temp_id1"] = df["register"] + "_" + df["temp_id1"]

     # --- ID temporaire : entier séquentiel simple, 1 à N ---
    df = df.reset_index(drop=True)
    df["temp_id2"] = df.index + 1

 
    final_cols = [
        "temp_id2","temp_id1", "register", "folio_label", "folio_sort_key",
        "ark_id", "urlImage", "base_image", "image_path", "n_zones", "__source_file",
    ]
    return df[final_cols], unmatched_df


def print_unmatched_images(df: pd.DataFrame) -> pd.DataFrame:
    """Affiche origine et identité des images qui n'ont pas trouvé d'URL physique."""
    unmatched = df[df["urlImage"].isna() | (df["urlImage"] == "")]
    cols = [c for c in ["register", "folio_label", "folio_sort_key", "ark_id",
                         "image_path", "__source_file"] if c in unmatched.columns]
    print(f"{len(unmatched)} image(s) sans correspondance :")
    print(unmatched[cols].to_string(index=False))
    return unmatched

In [111]:
zones_df = load_all_zone_exports(ZONES_INPUT_DIR, ZONES_GLOB_PATTERN)
zones_df = exclude_known_duplicate_images(zones_df, EXCLUDED_ARK_IDS)
check_missing_canonical_columns(zones_df)


images_df = pd.read_csv(IMAGES_CSV_PATH)

unique_images_df, unmatched_df = build_unique_images_from_zones(zones_df, images_df, include_unannotated=True)
print(f"\n{len(unique_images_df)} images conservées, {len(unmatched_df)} écartées (voir unmatched_df pour le détail).")

unique_images_df.to_csv("images_a_importer_heurist.csv", index=False, encoding="utf-8")
unique_images_df.head()

Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ100-JJ118_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ100-JJ118_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ119-JJ132_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ119-JJ132_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ133-JJ139_labelstudio.csv
  Normalisation colonnes (Himanis_Seg_Actes_1200pxmin_JJ133-JJ139_labelstudio.csv): {'ordre': 'folio_sort_key', 'registre': 'register'}
Lecture export LS: ../List-of-zones\Himanis_Seg_Actes_1200pxmin_JJ140-JJ159_labelstudio.

,temp_id2,temp_id1,register,folio_label,folio_sort_key,ark_id,urlImage,base_image,image_path,n_zones,__source_file
0,1,JJ096_0001,JJ096,plat sup��rieur,1,ark:/63955/vd0qz1ihxyni,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0746_A,images_registres_AN_JJ035_JJ211/images\Paris_A...,0,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
1,2,JJ096_0002,JJ096,contre-plat sup��rieur,2,ark:/63955/v6e576jlpbid,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0746_AB_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
2,3,JJ096_0003,JJ096,1r,3,ark:/63955/vxyc3rg33kh4,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0747_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
3,4,JJ096_0004,JJ096,1v,4,ark:/63955/vbg5uy3smlz0,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0748_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...
4,5,JJ096_0005,JJ096,2r,5,ark:/63955/vr66brwmaaj8,https://iiif.irht.cnrs.fr/iiif/VOLUME1/France/...,FRAN_0021_0749_A,images_registres_AN_JJ035_JJ211/images/Paris_A...,1,Himanis_Seg_Actes_1200pxmin_JJ096-JJ099_labels...


## Dimensions déclarées (jointure ark_id -> taille déclarée de l'image)

IIIF exprime toujours la région (`region` dans `/region/size/rotation/quality.format`) en coordonnées de l'image **pleine résolution déclarée** — indépendamment de la taille réellement demandée dans l'URL (ex. `/full/1200,/...`). On utilise donc `imageWidthAsDeclared` / `imageHeightAsDeclared` (liste d'images), pas la taille téléchargée ni `original_width`/`original_height` du JSON Label Studio (qui reflète seulement la taille d'affichage pendant l'annotation).

In [112]:
def build_declared_dims_lookup(images_df: pd.DataFrame) -> dict:
    """
    Construit un dict ark_id -> (declared_width, declared_height) à partir de la liste
    d'images (fichier 3), en utilisant la TAILLE DÉCLARÉE (imageWidthAsDeclared /
    imageHeightAsDeclared) — PAS la taille téléchargée ni la taille d'affichage
    Label Studio (original_width/height du JSON). C'est essentiel : IIIF exprime
    toujours la région ('region' dans /region/size/rotation/quality.format) en
    coordonnées de l'image PLEINE RÉSOLUTION déclarée, quelle que soit la taille
    demandée dans l'URL (ex. .../full/1200,/... peut très bien nécessiter une
    région du type 3000,2000,1000,1000 si l'image déclarée est bien plus grande).
    """
    df = images_df.copy()
    df["ark_id"] = df["urlImage_arkId"].apply(extract_ark_id)
    df["imageWidthAsDeclared"] = pd.to_numeric(df["imageWidthAsDeclared"], errors="coerce")
    df["imageHeightAsDeclared"] = pd.to_numeric(df["imageHeightAsDeclared"], errors="coerce")
    df = df.dropna(subset=["ark_id", "imageWidthAsDeclared", "imageHeightAsDeclared"])
    df = df.drop_duplicates(subset=["ark_id"])

    lookup = {
        row.ark_id: (row.imageWidthAsDeclared, row.imageHeightAsDeclared)
        for row in df.itertuples(index=False)
    }
    print(f"Dimensions déclarées disponibles pour {len(lookup)} image(s) (ark_id unique).")
    return lookup


## Conversion des coordonnées : Label Studio (%) → IIIF (pixels, taille déclarée)

⚠️ **Point à valider** : d'après la documentation Label Studio et votre propre code dans `JJ100-139-versImportLabelStudio2.ipynb` (cellule 29, commentaire *"Label Studio : x,y coin supérieur gauche"*), le format `x,y` de Label Studio pour `RectangleLabels` est le **coin supérieur gauche** (top-left) — la rotation, toujours à 0 dans vos exports, pivote autour de ce même coin. Le format « centre x,y » correspond au format **YOLO** (utilisé côté pré-annotations), pas à l'export Label Studio lui-même.

La fonction ci-dessous accepte les deux (`origin="top_left"` ou `"center"`), et la cellule de contrôle qui suit calcule les deux sur un cas réel quasi pleine-page : le bon format doit donner des coordonnées `ulx`/`uly` proches de 0.

In [113]:
def convert_ls_pct_to_iiif_px(x_pct, y_pct, w_pct, h_pct,
                               declared_w, declared_h, origin="top_left"):
    """
    Convertit un rectangle Label Studio (pourcentages) en coordonnées IIIF pixel
    (coin supérieur gauche + largeur/hauteur), sur la base de la taille DÉCLARÉE
    de l'image.

    origin="top_left" (par défaut) : x_pct/y_pct sont déjà le coin supérieur
        gauche du rectangle — c'est le format documenté par Label Studio pour
        RectangleLabels (rotation appliquée autour de ce même coin), et c'est
        aussi ce que suppose votre propre code dans
        JJ100-139-versImportLabelStudio2.ipynb (cellule 29 : "Label Studio :
        x,y coin supérieur gauche").
    origin="center" : x_pct/y_pct sont interprétés comme le CENTRE du
        rectangle (format YOLO). À utiliser seulement si la vérification
        visuelle (cellule de contrôle ci-dessous) confirme que c'est le bon
        format pour vos exports.

    Retourne (ulx_px, uly_px, w_px, h_px), arrondis à l'entier.
    """
    if origin == "top_left":
        ulx_pct, uly_pct = x_pct, y_pct
    elif origin == "center":
        ulx_pct, uly_pct = x_pct - w_pct / 2, y_pct - h_pct / 2
    else:
        raise ValueError(f"origin inconnu: {origin!r} (attendu 'top_left' ou 'center')")

    ulx_px = round(ulx_pct / 100 * declared_w)
    uly_px = round(uly_pct / 100 * declared_h)
    w_px = round(w_pct / 100 * declared_w)
    h_px = round(h_pct / 100 * declared_h)
    return ulx_px, uly_px, w_px, h_px


def build_iiif_region_url(physical_url: str, ulx: int, uly: int, w: int, h: int,
                           size_suffix: str = "600,", rotation: str = "0",
                           quality_format: str = "default.jpg") -> str:
    """
    Reconstruit une URL IIIF de région à partir de l'URL image physique pleine
    (.../{region}/{size}/{rotation}/{quality}.{ext}), en remplaçant uniquement
    la région par des coordonnées pixel absolues (coin sup. gauche + largeur/hauteur),
    et en fixant taille/rotation/qualité aux valeurs voulues pour la vignette.
    """
    root = "/".join(str(physical_url).rstrip("/").split("/")[:-4])
    region = f"{ulx},{uly},{w},{h}"
    return f"{root}/{region}/{size_suffix}/{rotation}/{quality_format}"


# --- Cellule de contrôle : top_left vs center, sur un cas réel quasi pleine-page ---
# Exemple tiré de vos données (folio 1r, zone AI quasi pleine page) :
#   x=0.485, y=0.18, width=99.51, height=99.82  (image déclarée ex. 1200x1422)
_test_declared_w, _test_declared_h = 1200, 1422
for _origin in ("top_left", "center"):
    _ulx, _uly, _w, _h = convert_ls_pct_to_iiif_px(
        0.485, 0.18, 99.51, 99.82, _test_declared_w, _test_declared_h, origin=_origin
    )
    print(f"origin={_origin:9s} -> ulx={_ulx:>6} uly={_uly:>6} w={_w:>6} h={_h:>6}"
          f"  (image {_test_declared_w}x{_test_declared_h})")
# Attendu pour une zone quasi pleine page : ulx et uly proches de 0.
# Si 'center' donne un ulx/uly fortement négatif, c'est le signe que 'top_left' est le bon format.


origin=top_left  -> ulx=     6 uly=     3 w=  1194 h=  1419  (image 1200x1422)
origin=center    -> ulx=  -591 uly=  -707 w=  1194 h=  1419  (image 1200x1422)


## Construction de la table des zones — tout le corpus en une seule fois

Une ligne par rectangle annoté, avec :
- coordonnées IIIF (`abs_x`=ulx, `abs_y`=uly, `abs_w`, `abs_h`) en pixels, taille déclarée ;
- tri vertical (haut → bas, puis gauche → droite) au sein de chaque image ;
- `zone_label` lisible (`z1` à `zN`, remis à zéro par image) + `zone_temp_id` unique sur tout le corpus (clé primaire pour l'import Heurist et les jointures ultérieures) ;
- `folio_label`, `folio_sort_key`, `register`, `base_image`, `urlImage` (physique) rattachés via l'ark_id de l'image.

In [114]:
def explode_all_zones(zones_df: pd.DataFrame, images_df: pd.DataFrame,
                       unique_images_df: pd.DataFrame,
                       origin: str = "top_left",
                       thumbnail_size: str = "600,") -> pd.DataFrame:
    """
    Explose la colonne JSON 'label' de chaque ligne de zones_df (une image) en une
    ligne par rectangle annoté, sur l'ensemble du corpus en une seule fois.

    - Coordonnées IIIF (ulx, uly, w, h) calculées en pixels sur la TAILLE DÉCLARÉE
      de l'image (voir build_declared_dims_lookup / convert_ls_pct_to_iiif_px).
    - Tri vertical (haut -> bas, puis gauche -> droite) au sein de chaque image.
    - Identifiant de zone lisible 'zone_label' (z1 à zN, remis à zéro à chaque image)
      + 'zone_temp_id' unique sur tout le corpus (clé primaire pour Heurist / jointures).
    - folio_label, folio_sort_key, register, base_image, urlImage (physique) rattachés
      à chaque zone via l'ark_id de l'image.
    """
    dims_lookup = build_declared_dims_lookup(images_df)

    meta_lookup = (
        unique_images_df[["ark_id", "temp_id2", "register", "folio_label",
                           "folio_sort_key", "base_image", "urlImage"]]
        .drop_duplicates(subset=["ark_id"])
        .set_index("ark_id")
    )

    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)

    zone_records = []
    n_parse_errors = 0
    n_missing_dims = 0
    n_missing_meta = 0
    n_images_with_zones = 0

    for row in df.itertuples(index=False):
        ark_id = getattr(row, "ark_id")
        label_raw = getattr(row, "label")
        if not isinstance(label_raw, str) or not label_raw.strip():
            continue

        try:
            rectangles = json.loads(label_raw)
        except (json.JSONDecodeError, TypeError):
            n_parse_errors += 1
            continue
        if not isinstance(rectangles, list) or not rectangles:
            continue

        if ark_id not in dims_lookup:
            n_missing_dims += 1
            continue
        declared_w, declared_h = dims_lookup[ark_id]

        if ark_id not in meta_lookup.index:
            n_missing_meta += 1
            continue
        meta = meta_lookup.loc[ark_id]
        physical_url = meta["urlImage"]

        bboxes = []
        for rect in rectangles:
            labels = rect.get("rectanglelabels")
            if not labels:
                continue
            x_pct, y_pct = rect.get("x", 0), rect.get("y", 0)
            w_pct, h_pct = rect.get("width", 0), rect.get("height", 0)
            ulx, uly, w_px, h_px = convert_ls_pct_to_iiif_px(
                x_pct, y_pct, w_pct, h_pct, declared_w, declared_h, origin=origin
            )
            bboxes.append({
                "class_name": labels[0],
                "abs_x": ulx, "abs_y": uly, "abs_w": w_px, "abs_h": h_px,
                "x_pct": x_pct, "y_pct": y_pct, "w_pct": w_pct, "h_pct": h_pct,
            })

        if not bboxes:
            continue

        # Tri vertical (haut -> bas), puis horizontal (gauche -> droite)
        bboxes.sort(key=lambda z: (z["abs_y"], z["abs_x"]))
        n_images_with_zones += 1

        for order, z in enumerate(bboxes, start=1):
            crop_url = build_iiif_region_url(
                physical_url, z["abs_x"], z["abs_y"], z["abs_w"], z["abs_h"],
                size_suffix=thumbnail_size,
            )
            zone_records.append({
                "zone_label": f"z{order}",
                "zone_order_on_folio": order,
                "image_ark_id": ark_id,
                "image_temp_id": meta["temp_id2"],
                "register": meta["register"],
                "folio_label": meta["folio_label"],
                "folio_sort_key": meta["folio_sort_key"],
                "class_name": z["class_name"],
                "abs_x": z["abs_x"], "abs_y": z["abs_y"],
                "abs_w": z["abs_w"], "abs_h": z["abs_h"],
                "x_pct": z["x_pct"], "y_pct": z["y_pct"],
                "w_pct": z["w_pct"], "h_pct": z["h_pct"],
                "declared_width": declared_w, "declared_height": declared_h,
                "base_image": meta["base_image"],
                "urlImage": physical_url,
                "address_bvmm_path": crop_url,
                "address_bvmm_name": "_remote",
                "__source_file": getattr(row, "__source_file", None),
            })

    zones_out = pd.DataFrame(zone_records)
    if not zones_out.empty:
        # Tri numérique sur folio_sort_key (colonne texte à l'origine) sans
        # modifier son dtype d'affichage : clé de tri temporaire uniquement.
        zones_out["__folio_sort_key_num"] = pd.to_numeric(
            zones_out["folio_sort_key"], errors="coerce"
        )
        zones_out = zones_out.sort_values(
            ["register", "__folio_sort_key_num", "zone_order_on_folio"],
            kind="mergesort",  # tri stable : préserve l'ordre relatif à égalité de clé
        ).drop(columns="__folio_sort_key_num").reset_index(drop=True)
    zones_out.insert(0, "zone_temp_id", zones_out.index + 1)  # index refait sur le tri correct

    print(f"{len(zones_out)} zones extraites depuis {n_images_with_zones} image(s) annotée(s) "
          f"(sur {len(df)} lignes lues).")
    if n_parse_errors:
        print(f"⚠️  {n_parse_errors} ligne(s) avec un JSON 'label' invalide (ignorées).")
    if n_missing_dims:
        print(f"⚠️  {n_missing_dims} ligne(s) sans dimensions déclarées (zones ignorées).")
    if n_missing_meta:
        print(f"⚠️  {n_missing_meta} ligne(s) sans métadonnées image (zones ignorées).")

    return zones_out


In [115]:
def print_missing_dims_images(zones_df: pd.DataFrame, images_df: pd.DataFrame) -> pd.DataFrame:
    """Identifie les ark_id de zones_df absents de dims_lookup (pas de dimensions déclarées)."""
    dims_lookup = build_declared_dims_lookup(images_df)

    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)

    # Une ligne par image annotée (pas par zone), pour ne pas dupliquer le diagnostic
    df_annotated = df[df["label"].astype(str).str.strip() != ""]
    missing = df_annotated[~df_annotated["ark_id"].isin(dims_lookup.keys())].drop_duplicates(subset=["ark_id"])

    cols = [c for c in ["register", "ark_id", "image_path", "folio_sort_key", "__source_file"] if c in missing.columns]
    print(f"{len(missing)} image(s) annotée(s) sans dimensions déclarées :")
    print(missing[cols].to_string(index=False))
    return missing


df_missing_dims = print_missing_dims_images(zones_df, images_df)

zones_exploded_df = explode_all_zones(
    zones_df, images_df, unique_images_df,
    origin="top_left",       # <- à repasser à "center" seulement si la cellule de contrôle le confirme
    thumbnail_size="600,",
)
zones_exploded_df.to_csv("zones_a_importer_heurist.csv", index=False, encoding="utf-8")
zones_exploded_df.head(10)


def check_zones_lost_on_missing_dims(zones_df: pd.DataFrame, images_df: pd.DataFrame) -> pd.DataFrame:
    """
    Pour les images annotées absentes de dims_lookup (donc écartées par explode_all_zones),
    compte le nombre de zones qu'elles contenaient réellement dans le JSON 'label',
    pour vérifier si des zones sont perdues silencieusement.
    """
    dims_lookup = build_declared_dims_lookup(images_df)

    df = zones_df.copy()
    df["ark_id"] = df["image"].apply(extract_ark_id)
    df_annotated = df[df["label"].astype(str).str.strip() != ""].copy()

    missing = df_annotated[~df_annotated["ark_id"].isin(dims_lookup.keys())].drop_duplicates(subset=["ark_id"]).copy()

    def count_rects(label_raw):
        try:
            data = json.loads(label_raw)
            return len(data) if isinstance(data, list) else 0
        except (json.JSONDecodeError, TypeError):
            return -1  # JSON invalide

    missing["n_zones_perdues"] = missing["label"].apply(count_rects)

    cols = [c for c in ["register", "ark_id", "image_path", "folio_sort_key",
                         "n_zones_perdues", "__source_file"] if c in missing.columns]
    total_perdues = missing["n_zones_perdues"].clip(lower=0).sum()
    print(f"{len(missing)} image(s) écartée(s) faute de dimensions déclarées, "
          f"totalisant {total_perdues} zone(s) potentiellement perdue(s) :")
    print(missing[cols].to_string(index=False))
    return missing


df_zones_lost = check_zones_lost_on_missing_dims(zones_df, images_df)

Dimensions déclarées disponibles pour 49890 image(s) (ark_id unique).
0 image(s) annotée(s) sans dimensions déclarées :
Empty DataFrame
Columns: [register, ark_id, image_path, folio_sort_key, __source_file]
Index: []
Dimensions déclarées disponibles pour 49890 image(s) (ark_id unique).
83966 zones extraites depuis 49360 image(s) annotée(s) (sur 49850 lignes lues).
Dimensions déclarées disponibles pour 49890 image(s) (ark_id unique).
0 image(s) écartée(s) faute de dimensions déclarées, totalisant 0 zone(s) potentiellement perdue(s) :
Empty DataFrame
Columns: [register, ark_id, image_path, folio_sort_key, n_zones_perdues, __source_file]
Index: []


## Vérifications de cohérence

Logiques portées et généralisées depuis `JJ096-211-act-image-zone.ipynb`, appliquées ici sur l'ensemble du corpus (`register`/`folio_sort_key` plutôt que par sous-corpus séparé).

### A. Folios manquants dans la liste d'images (trous de numérotation par registre)

In [116]:
def find_missing_folios(images_df_with_sortkey: pd.DataFrame) -> pd.DataFrame:
    """
    Détecte les 'trous' dans la numérotation séquentielle des folios (folio_sort_key)
    par registre — utile pour repérer des images manquantes dans la liste d'images,
    PAS seulement des images non annotées (on l'applique donc sur la liste d'images
    complète, unique_images_df, qui inclut aussi les images sans zone).
    """
    missing = []
    df = images_df_with_sortkey.dropna(subset=["folio_sort_key"]).copy()
    df["folio_sort_key"] = pd.to_numeric(df["folio_sort_key"], errors="coerce")
    df = df.dropna(subset=["folio_sort_key"])

    for register, g in df.groupby("register"):
        folios = sorted(g["folio_sort_key"].unique())
        expected = set(range(int(min(folios)), int(max(folios)) + 1))
        actual = set(int(f) for f in folios)
        gaps = sorted(expected - actual)
        if gaps:
            missing.append({"register": register, "missing_folio_sort_keys": gaps, "count": len(gaps)})

    return pd.DataFrame(missing)


df_missing_folios = find_missing_folios(unique_images_df)
print(f"Registres avec trous dans la numérotation des folios : {len(df_missing_folios)}")
df_missing_folios


Registres avec trous dans la numérotation des folios : 0


""


### B. Zones superposées (>50% de recouvrement vertical) au sein d'une même image

In [117]:
def y_overlap_ratio(y1, h1, y2, h2):
    """
    Taux de recouvrement vertical entre deux zones, en proportion de la hauteur
    de la plus petite des deux. 0.0 = pas de recouvrement, 1.0 = une zone
    contient totalement l'autre.
    """
    top = max(y1, y2)
    bottom = min(y1 + h1, y2 + h2)
    overlap = max(0, bottom - top)
    if overlap == 0:
        return 0.0
    smaller_h = min(h1, h2)
    return overlap / smaller_h if smaller_h > 0 else 0.0


def find_overlapping_zones(df: pd.DataFrame, threshold: float = 0.5) -> pd.DataFrame:
    """
    Détecte les paires de zones superposées à plus de `threshold` sur l'axe Y,
    au sein d'une même image (register, folio_sort_key).
    """
    conflicts = []

    for (register, folio), g in df.groupby(["register", "folio_sort_key"]):
        g = g.sort_values("abs_y").reset_index(drop=True)
        zones = g.to_dict("records")

        for i in range(len(zones)):
            for j in range(i + 1, len(zones)):
                z1, z2 = zones[i], zones[j]
                if pd.isna(z1["abs_y"]) or pd.isna(z1["abs_h"]) \
                   or pd.isna(z2["abs_y"]) or pd.isna(z2["abs_h"]):
                    continue
                if z2["abs_y"] > z1["abs_y"] + z1["abs_h"]:
                    break
                ratio = y_overlap_ratio(z1["abs_y"], z1["abs_h"], z2["abs_y"], z2["abs_h"])
                if ratio > threshold:
                    conflicts.append({
                        "register": register,
                        "folio_sort_key": folio,
                        "folio_label": z1.get("folio_label"),
                        "zone_1": z1["zone_label"], "zone_temp_id_1": z1["zone_temp_id"],
                        "class_name_1": z1["class_name"],
                        "zone_2": z2["zone_label"], "zone_temp_id_2": z2["zone_temp_id"],
                        "class_name_2": z2["class_name"],
                        "overlap_ratio": round(ratio, 2),
                    })

    return pd.DataFrame(conflicts)


df_overlaps = find_overlapping_zones(zones_exploded_df, threshold=0.5)
print(f"Zones superposées (>50% axe Y) : {len(df_overlaps)}")
df_overlaps.head(20)


Zones superposées (>50% axe Y) : 210


,register,folio_sort_key,folio_label,zone_1,zone_temp_id_1,class_name_1,zone_2,zone_temp_id_2,class_name_2,overlap_ratio
0,JJ099,276,130v,z3,3376,AC,z4,3377,AC,1.00
1,JJ099,276,130v,z3,3376,AC,z5,3378,AC,0.96
2,JJ099,276,130v,z4,3377,AC,z5,3378,AC,0.98
3,JJ099,276,130v,z6,3379,AC,z7,3380,AC,1.00
4,JJ106,120,59v,z1,8433,AF,z2,8434,AC,0.99
5,JJ106,120,59v,z1,8433,AF,z3,8435,AC,0.55
6,JJ106,120,59v,z2,8434,AC,z3,8435,AC,0.58
7,JJ106,120,59v,z3,8435,AC,z4,8436,AC,0.83
8,JJ106,183,91r,z1,8549,AF,z2,8550,AC,1.00
9,JJ108,22,10v,z1,9705,AF,z2,9706,AC,0.85


### C. Validation de la structure de séquence des zones par folio (`[AC]+` / `[AM]` seul / `[NIA]` seul / `[Table]` seul / `(AF)? AC* (AI)?`)

In [118]:
def check_vertical_order(df: pd.DataFrame) -> pd.DataFrame:
    """
    Contrôle vectorisé : dans chaque folio (register, folio_sort_key), les zones
    doivent être ordonnées de haut en bas (abs_y croissant) selon
    zone_order_on_folio. Signale toute rupture (utile si zones_exploded_df a été
    retrié ou modifié après sa création par explode_all_zones, qui garantit
    normalement déjà cet ordre).
    """
    d = df.copy()
    d["folio_sort_key"] = pd.to_numeric(d["folio_sort_key"], errors="coerce")
    d = d.sort_values(["register", "folio_sort_key", "zone_order_on_folio"])

    same_folio = (
        (d["register"] == d["register"].shift())
        & (d["folio_sort_key"] == d["folio_sort_key"].shift())
    )
    y_decreases = d["abs_y"] < d["abs_y"].shift()
    bad = d.loc[
        same_folio & y_decreases,
        ["register", "folio_sort_key", "folio_label", "zone_order_on_folio", "abs_y"],
    ]
    return bad


df_order_issues = check_vertical_order(zones_exploded_df)
print(f"Ruptures d'ordre vertical détectées : {len(df_order_issues)}")
df_order_issues.head(20)


Ruptures d'ordre vertical détectées : 0


,register,folio_sort_key,folio_label,zone_order_on_folio,abs_y


###  Corrections manuelles au niveau zone

Registre indépendant (`zone_corrections.csv`) pour documenter les anomalies
identifiées lors des contrôles de séquence/transition, sans jamais modifier
`zones_exploded_df` ni le CSV source. Les corrections sont appliquées à la
volée (`apply_zone_corrections`) pour produire une vue corrigée utilisée par
les contrôles suivants, et resteront exploitables plus tard pour la liaison
zones ↔ actes.


In [119]:


# ============================================================
# CORRECTIONS MANUELLES
# ============================================================

ZONE_CORRECTIONS_PATH = "zone_corrections.csv"

ZONE_CORRECTION_COLUMNS = [
    "register",
    "folio_sort_key",
    "folio_label",
    "zone_temp_id",
    "action",
    "target_zone_temp_id",
    "value",
    "note",
]

VALID_ACTIONS = {
    "changer_classe",
    "exclure_zone",
    "suit_zone",
    "precede_zone",
}


def load_zone_corrections(
    path: str = ZONE_CORRECTIONS_PATH
) -> pd.DataFrame:
    """Charge le CSV des corrections manuelles."""
    if not os.path.exists(path):
        return pd.DataFrame(columns=ZONE_CORRECTION_COLUMNS)
    corrections = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        na_values=[""],
    )
    # Évite les erreurs dues à des espaces accidentels
    # dans les noms de colonnes.
    corrections.columns = corrections.columns.str.strip()

    # Idem pour les valeurs elles-mêmes (ex. "exclure_zone " avec espace
    # final, fréquent après copier-coller depuis Excel) : sans ce nettoyage,
    # les comparaisons comme action == "exclure_zone" échouent silencieusement.
    for col in corrections.columns:
        corrections[col] = corrections[col].str.strip()

    # Détection des actions non reconnues (typo, valeur non nettoyée, etc.)
    unknown = set(corrections.loc[corrections["action"] != "", "action"]) - VALID_ACTIONS
    if unknown:
        print(f"⚠️  action(s) inconnue(s) dans {path}, ignorées : {sorted(unknown)}")

    return corrections



# ============================================================
# APPLICATION DES CORRECTIONS
# ============================================================

def apply_zone_corrections(
    df: pd.DataFrame,
    path: str = ZONE_CORRECTIONS_PATH,
) -> pd.DataFrame:
    """
    Applique les corrections manuelles au DataFrame.

    Actions prises en charge :

    - changer_classe :
        change class_name.

    - exclure_zone :
        retire la zone de la séquence.

    - suit_zone :
        retire zone_temp_id de sa position actuelle
        et la place immédiatement après target_zone_temp_id.

    - precede_zone :
        retire zone_temp_id de sa position actuelle
        et la place immédiatement avant target_zone_temp_id.

    L'ordre global des lignes du DataFrame est l'ordre de référence.
    Les notions de registre et de folio ne sont plus utilisées
    pour la validation de séquence.
    """

    corrections = load_zone_corrections(path)

    d = df.copy()

    # Identifiants homogènes
    d["zone_temp_id"] = d["zone_temp_id"].astype(str)

    # ========================================================
    # 1. CHANGER LA CLASSE
    # ========================================================

    for _, r in corrections[
        corrections["action"] == "changer_classe"
    ].iterrows():

        zone_id = str(r["zone_temp_id"])

        mask = d["zone_temp_id"] == zone_id

        d.loc[mask, "class_name"] = r["value"]

    
    # ========================================================
    # 3. TRANSFORMER LE DATAFRAME EN LISTE DE LIGNES
    # ========================================================
    #
    # Cela permet de déplacer réellement une zone dans
    # la séquence globale.
    #

    rows = d.to_dict("records")

    # --------------------------------------------------------
    # Fonction utilitaire : trouver une zone
    # --------------------------------------------------------

    def find_index(rows, zone_id):
        zone_id = str(zone_id)

        for i, row in enumerate(rows):
            if str(row["zone_temp_id"]) == zone_id:
                return i

        return None

    # --------------------------------------------------------
    # Déplacer après une autre zone
    # --------------------------------------------------------

    def move_after(rows, zone_id, target_id):

        zone_id = str(zone_id)
        target_id = str(target_id)

        if zone_id == target_id:
            return rows

        zone_idx = find_index(rows, zone_id)
        target_idx = find_index(rows, target_id)

        if zone_idx is None:
            print(
                f"⚠️ suit_zone : zone introuvable "
                f"zone_temp_id={zone_id}"
            )
            return rows

        if target_idx is None:
            print(
                f"⚠️ suit_zone : cible introuvable "
                f"target_zone_temp_id={target_id}"
            )
            return rows

        # Retirer la zone
        zone = rows.pop(zone_idx)

        # Rechercher à nouveau la cible car son index
        # peut avoir changé après le retrait.
        target_idx = find_index(rows, target_id)

        # Insérer immédiatement après la cible
        rows.insert(target_idx + 1, zone)

        return rows

    # --------------------------------------------------------
    # Déplacer avant une autre zone
    # --------------------------------------------------------

    def move_before(rows, zone_id, target_id):

        zone_id = str(zone_id)
        target_id = str(target_id)

        if zone_id == target_id:
            return rows

        zone_idx = find_index(rows, zone_id)
        target_idx = find_index(rows, target_id)

        if zone_idx is None:
            print(
                f"⚠️ precede_zone : zone introuvable "
                f"zone_temp_id={zone_id}"
            )
            return rows

        if target_idx is None:
            print(
                f"⚠️ precede_zone : cible introuvable "
                f"target_zone_temp_id={target_id}"
            )
            return rows

        # Retirer la zone
        zone = rows.pop(zone_idx)

        # Rechercher à nouveau la cible
        target_idx = find_index(rows, target_id)

        # Insérer immédiatement avant la cible
        rows.insert(target_idx, zone)

        return rows

    # ========================================================
    # 4. APPLIQUER LES CORRECTIONS D'ORDRE
    # ========================================================

    for _, r in corrections.iterrows():

        action = r["action"]

        if action == "suit_zone":

            target_id = r["target_zone_temp_id"]

            if target_id:
                rows = move_after(
                    rows,
                    r["zone_temp_id"],
                    target_id,
                )

        elif action == "precede_zone":

            target_id = r["target_zone_temp_id"]

            if target_id:
                rows = move_before(
                    rows,
                    r["zone_temp_id"],
                    target_id,
                )

    # ========================================================
    # 5. RECONSTRUIRE LE DATAFRAME
    # ========================================================

    d = pd.DataFrame(rows).reset_index(drop=True)

    return d


# ============================================================
# APPLICATION DES CORRECTIONS
# ============================================================

df_corrected = apply_zone_corrections(
    zones_exploded_df,
    ZONE_CORRECTIONS_PATH,
)

# Export de la vue réellement utilisée pour la validation
CORRECTED_ZONES_PATH = "zones_exploded_corrige.csv"

df_corrected.to_csv(
    CORRECTED_ZONES_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Vue corrigée exportée vers : {CORRECTED_ZONES_PATH}")
print(f"Nombre de lignes : {len(df_corrected)}")



# ============================================================
# VALIDATION DE LA SÉQUENCE GLOBALE
# ============================================================
def validate_global_sequence(
        df: pd.DataFrame,
        corrections_path: str = ZONE_CORRECTIONS_PATH,
        ) -> pd.DataFrame:
    corrections = load_zone_corrections(corrections_path)
    excluded_ids = set(
        corrections.loc[
            corrections["action"] == "exclure_zone",
            "zone_temp_id",
        ].astype(str)
    )

    anomalies = []
    state = "normal"

    for position, (_, row) in enumerate(df.iterrows()):
        zone_id = str(row["zone_temp_id"])
        classe = row["class_name"]
        register = row["register"]
        folio_sort_key = row["folio_sort_key"]
        folio_label = row["folio_label"]

        def add_anomaly(reason):
            # La zone garde son rôle dans la machine à états (elle continue
            # d'interrompre/relancer la suite normalement) ; seule sa
            # présence dans la LISTE affichée est supprimée si elle est
            # explicitement exclue.
            if zone_id not in excluded_ids:
                anomalies.append({
                    "position": position, "register": register,
                    "folio_sort_key": folio_sort_key, "folio_label": folio_label,
                    "zone_temp_id": zone_id, "class_name": classe,
                    "reason": reason,
                })

        # ====================================================
        # ÉTAT NORMAL
        # ====================================================
        if state == "normal":
            if classe == "AC":
                continue
            elif classe == "AI":
                state = "suite"
                continue
            elif classe in {"NIA", "Table"}:
                continue
            elif classe == "AM":
                add_anomaly("AM sans AI préalable")
            elif classe == "AF":
                add_anomaly("AF sans AI préalable")
            else:
                add_anomaly("Classe inconnue")

        # ====================================================
        # SUITE AI ... AF EN COURS
        # ====================================================
        elif state == "suite":
            if classe == "AM":
                continue
            elif classe == "AF":
                state = "normal"
                continue
            elif classe == "AI":
                add_anomaly("Deuxième AI avant AF")
            elif classe == "AC":
                add_anomaly("AC interrompt une suite AI...AF")
            elif classe in {"NIA", "Table"}:
                add_anomaly(f"{classe} interrompt une suite AI...AF")
                state = "interrompue"
            else:
                add_anomaly("Classe inconnue dans une suite AI...AF")

        # ====================================================
        # SUITE INTERROMPUE PAR NIA / TABLE
        # ====================================================
        elif state == "interrompue":
            if classe in {"NIA", "Table"}:
                continue
            elif classe == "AF":
                state = "normal"
                continue
            elif classe == "AI":
                state = "suite"
                continue
            elif classe == "AC":
                state = "normal"
                continue
            else:
                add_anomaly(f"{classe} après interruption de la suite AI...AF")

    # ========================================================
    # FIN DE LA SÉQUENCE — une seule fois, hors boucle
    # ========================================================
    if state in {"suite", "interrompue"}:
        # Pas de zone_id précis ici : rien à exclure, mais on pourrait
        # vouloir l'ignorer aussi si la dernière zone réelle est exclue —
        # cas limite volontairement non traité pour l'instant.
        anomalies.append({
            "position": len(df), "register": None,
            "folio_sort_key": None, "folio_label": None,
            "zone_temp_id": None, "class_name": None,
            "reason": "Suite AI...AF non terminée : AF manquant",
        })

    return pd.DataFrame(anomalies)

# ============================================================
# EXÉCUTION
# ============================================================

df_corrected = apply_zone_corrections(
    zones_exploded_df,
    ZONE_CORRECTIONS_PATH,
)

df_anomalies = validate_global_sequence(
    df_corrected,
)

print(f"Nombre d'anomalies : {len(df_anomalies)}")

# Source - https://stackoverflow.com/q/16424493
# Posted by Andy, modified by community. See post 'Timeline' for change history
# Retrieved 2026-07-30, License - CC BY-SA 3.0

pd.set_option('display.max_rows', 500)

display(df_anomalies.head(10))



Vue corrigée exportée vers : zones_exploded_corrige.csv
Nombre de lignes : 83966
Nombre d'anomalies : 199


,position,register,folio_sort_key,folio_label,zone_temp_id,class_name,reason
0,13964,JJ113,365,contre-plat inférieur,13965,NIA,NIA interrompt une suite AI...AF
1,15292,JJ116,12,5v,15293,NIA,NIA interrompt une suite AI...AF
2,17570,JJ119,196,97v,17571,NIA,NIA interrompt une suite AI...AF
3,18238,JJ120,61,30r,18239,AC,AC interrompt une suite AI...AF
4,18239,JJ120,62,30v,18240,AC,AC interrompt une suite AI...AF
5,18240,JJ120,63,31r,18241,AI,Deuxième AI avant AF
6,19903,JJ122,244,122v,19904,NIA,NIA interrompt une suite AI...AF
7,20112,JJ122,357,179r,20113,AI,Deuxième AI avant AF
8,23607,JJ129,3,Ir,23608,AC,AC interrompt une suite AI...AF
9,23608,JJ129,4,Iv,23609,Table,Table interrompt une suite AI...AF


# Acte-Zones

Une fois ces contrôles passés en revue (en particulier le choix `origin` pour les coordonnées, et les éventuels folios/séquences/transitions signalés ci-dessus), l'étape suivante est la liaison `zones_exploded_df` ↔ liste des Actes, en s'appuyant sur `zone_label`/`zone_order_on_folio` pour apparier chaque acte à sa/ses zone(s) sur le(s) folio(s) concerné(s) (logique de rôles AI/AM/AF/AC déjà présente dans `JJ096-211-act-image-zone.ipynb`, à généraliser de la même façon sur tout le corpus).

## Paramètres

In [ ]:
ACTS_LIST_PATH = "../List-of-acts/Acts-JJ96-JJ211.csv"
ZONES_CORRIGE_PATH = "zones_exploded_corrige.csv"
MAX_REGISTER = "JJ180"  # limite provisoire, inclus

ACTE_CORRECTIONS_PATH = "acte_zone_corrections.csv"
ACTE_CORRECTION_COLUMNS = ["id_temp", "register", "act_number", "action",
                           "anchor_zone_temp_id", "zone_temp_id", "note"]
ACTE_VALID_ACTIONS = {"set_start_zone", "add_zone"}

"""
colonne	rôle
id_temp	ID-temporaire de l'acte concerné (informatif/traçabilité)
register, act_number	contexte (informatif)
action	set_start_zone | add_zone
anchor_zone_temp_id	pour add_zone uniquement : un zone_temp_id déjà connu de l'acte cible (typiquement sa zone de départ)
zone_temp_id	zone concernée (départ vérifié pour set_start_zone ; zone à rattacher pour add_zone)
note	commentaire libre
"""

# Règle spécifique JJ126 (et JJ181, même particularité) : les folios image
# sont zéro-préfixés ("001r"), contrairement aux folios acte ("1r"). On
# compare aussi les folio_norm en ignorant les zéros initiaux avant de
# conclure à un vrai décalage.
FOLIO_LEADING_ZEROS_REGISTERS = {"JJ126", "JJ181"}




## Fonctions

In [ ]:


def register_num(reg: str) -> int:
    m = re.search(r'(\d+)$', str(reg))
    return int(m.group(1)) if m else 10**6

def normalize_folio(raw):
    """
    Normalise un label de folio pour comparaison/jointure.
      '6'              -> '6r'
      '12v'            -> '12v'
      '103bis'         -> '103bisr'
      '103 bis recto'  -> '103bisr'
      'plat supérieur' -> 'plat supérieur'
    Repris de JJ096-211-act-image-zone.ipynb.
    """
    s = str(raw).strip().lower()

    if s in ('vacat', 'vacatr', 'vacatv'):
        return None
    if not s or s == 'nan':
        return None

    s = s.replace('recto', 'r').replace('verso', 'v')
    s = re.sub(r'\s+bis\s*', 'bis', s)
    s = re.sub(r'\s+([rv])$', r'\1', s)

    if re.match(r'^\d+(?:bis)?[rv]$', s):
        return s
    if re.match(r'^\d+(?:bis)?$', s):
        return s + 'r'
    return s

def load_acts_list(path: str = ACTS_LIST_PATH, max_register: str = MAX_REGISTER) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";", dtype=str, encoding="utf-8-sig",
                      keep_default_na=False, na_values=[""])
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].str.strip()

    # Détection insensible à la casse/espaces/tirets/underscores, comme pour
    # id_temp/register/act_number : évite les échecs silencieux de rename()
    # sur une orthographe légèrement différente de celle attendue.
    def norm_key(s):
        return s.lower().replace(" ", "").replace("_", "").replace("-", "")

    lookup = {norm_key(c): c for c in df.columns}

    rename_map = {}
    for target, candidates in {
        "id_temp": ["id-temporaire", "idtemporaire"],
        "register": ["register"],
        "act_number": ["act_number", "actnumber"],
        "folio_raw": ["folio", "foliolabel", "folionumberoupage"],
    }.items():
        found = next((lookup[c] for c in candidates if c in lookup), None)
        if found:
            rename_map[found] = target

    df = df.rename(columns=rename_map)

    missing_required = {"id_temp", "register", "act_number"} - set(df.columns)
    if missing_required:
        raise ValueError(f"Colonne(s) requise(s) introuvable(s) dans {path} : {missing_required}. "
                          f"Colonnes disponibles : {list(df.columns)}")

    df = df[df["register"].map(register_num) <= register_num(max_register)]
    df["excel_order"] = range(len(df))

    if "folio_raw" in df.columns:
        df["folio_norm"] = df["folio_raw"].apply(normalize_folio)
    else:
        print(f"⚠️  Colonne FOLIO introuvable dans {path} — vérification folio ignorée.")
        print(f"   Colonnes disponibles : {list(df.columns)}")

    return df.reset_index(drop=True)

IMAGES_LIST_PATH = "../List-of-images/JJ096-JJ211_image_data.csv"


def load_images_list(path: str = IMAGES_LIST_PATH, max_register: str = MAX_REGISTER) -> pd.DataFrame:
    """
    Charge (register, folio_label, urlImage), normalise folio_label pour
    servir de référence à la vérification du champ FOLIO des actes.
    """
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].str.strip()

    required = {"register", "folio_label", "urlImage"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes manquantes dans {path} : {missing}. "
                          f"Colonnes trouvées : {list(df.columns)}")

    df = df[df["register"].map(register_num) <= register_num(max_register)]
    df["folio_norm"] = df["folio_label"].apply(normalize_folio)
    return df.reset_index(drop=True)

def check_acte_folio_vs_images(df_actes: pd.DataFrame, df_images: pd.DataFrame) -> pd.DataFrame:
    """
    Vérifie que le folio déclaré pour chaque acte (colonne FOLIO, normalisé)
    correspond à une image existante du même registre (folio_label normalisé
    de la même façon). Ajoute folio_ok (bool) et folio_probleme (motif).
    """
    if "folio_norm" not in df_actes.columns:
        raise ValueError("df_actes n'a pas de colonne 'folio_norm' — "
                          "la colonne FOLIO est-elle présente dans Acts CSV ?")

    valid_folios = (
        df_images.dropna(subset=["folio_norm"])
                 .groupby("register")["folio_norm"]
                 .apply(set)
                 .to_dict()
    )

    def check(row):
        if not row["folio_norm"]:
            return "folio vide ou non reconnu"
        if row["folio_norm"] not in valid_folios.get(row["register"], set()):
            return f"folio '{row['folio_norm']}' absent des images de {row['register']}"
        return ""

    d = df_actes.copy()
    d["folio_probleme"] = d.apply(check, axis=1)
    d["folio_ok"] = d["folio_probleme"] == ""
    return d


def display_folio_check(df_checked: pd.DataFrame, cols=None):
    """Affiche le tableau avec les lignes incompatibles surlignées en rouge."""
    cols = cols or ["id_temp", "register", "act_number", "folio_raw", "folio_norm", "folio_probleme"]
    view = df_checked[cols]

    def highlight_bad(row):
        color = "background-color: #f8d7da" if row["folio_probleme"] else ""
        return [color] * len(row)

    return view.style.apply(highlight_bad, axis=1)


def load_zones_corrigees(path: str = ZONES_CORRIGE_PATH, max_register: str = MAX_REGISTER) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    df.columns = [c.strip() for c in df.columns]
    for c in ("zone_order_on_folio", "folio_sort_key", "x_pct", "y_pct", "w_pct", "h_pct"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df[df["register"].map(register_num) <= register_num(max_register)]
    # L'ordre des lignes du fichier EST l'ordre de lecture (tri + suit_zone/
    # precede_zone déjà appliqués en amont) : on ne retrie surtout pas ici.
    return df.reset_index(drop=True)

In [121]:
def norm_key(s):
    return s.lower().replace(" ", "").replace("_", "").replace("-", "")


In [122]:
def build_acte_groups(df_zones: pd.DataFrame) -> pd.DataFrame:
    """
    Regroupe les zones en actes, dans l'ordre de lecture :
      - AC : acte à zone unique (n'ouvre pas de suite)
      - AI : ouvre une suite, rattache AM/AF jusqu'à l'AF fermant
      - AM/AF sans acte ouvert : orphelines (orphan=True) -> candidates
        à un rattachement manuel via add_zone
      - NIA/Table : ferment l'acte ouvert (comme AF) ; le cas où l'acte
        reprend réellement après la NIA reste une exception à documenter
        explicitement via add_zone (acte_zone_corrections.csv), pas un
        comportement silencieux par défaut.
    """
    d = df_zones.copy()
    d["acte_seq_num"] = pd.NA
    d["acte_role"] = ""
    d["orphan"] = False

    for register, g in d.groupby("register", sort=False):
        seq_num = 0
        open_acte = False
        for idx in g.index:
            classe = d.at[idx, "class_name"]
            if classe == "AC":
                seq_num += 1
                d.at[idx, "acte_seq_num"] = seq_num
                d.at[idx, "acte_role"] = "start"
                open_acte = False
            elif classe == "AI":
                seq_num += 1
                open_acte = True
                d.at[idx, "acte_seq_num"] = seq_num
                d.at[idx, "acte_role"] = "start"
            elif classe in ("AM", "AF"):
                if open_acte:
                    d.at[idx, "acte_seq_num"] = seq_num
                    d.at[idx, "acte_role"] = "continuation"
                    if classe == "AF":
                        open_acte = False
                else:
                    d.at[idx, "orphan"] = True
            elif classe in ("NIA", "Table"):
                open_acte = False  # ferme l'acte en cours, comme AF
    return d

In [ ]:

def load_acte_corrections(path: str = ACTE_CORRECTIONS_PATH) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame(columns=ACTE_CORRECTION_COLUMNS)
    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].str.strip()
    unknown = set(df.loc[df["action"] != "", "action"]) - ACTE_VALID_ACTIONS
    if unknown:
        print(f"⚠️  action(s) inconnue(s) dans {path}, ignorées : {sorted(unknown)}")
    return df


def apply_acte_corrections(df_grouped: pd.DataFrame, path: str = ACTE_CORRECTIONS_PATH):
    corrections = load_acte_corrections(path)
    d = df_grouped.copy()
    d["zone_temp_id"] = d["zone_temp_id"].astype(str)
    zone_to_acte = d.set_index("zone_temp_id")["acte_seq_num"]

    for _, r in corrections[corrections["action"] == "add_zone"].iterrows():
        anchor_id, zone_id = str(r["anchor_zone_temp_id"]), str(r["zone_temp_id"])
        if anchor_id not in zone_to_acte.index or zone_id not in zone_to_acte.index:
            print(f"⚠️ add_zone : zone(s) introuvable(s) ({anchor_id} / {zone_id}) — {r['note']}")
            continue
        acte_num = zone_to_acte.loc[anchor_id]
        mask = d["zone_temp_id"] == zone_id
        d.loc[mask, "acte_seq_num"] = acte_num
        d.loc[mask, "acte_role"] = "continuation (ajoutée)"
        d.loc[mask, "orphan"] = False

    anchors = corrections[corrections["action"] == "set_start_zone"].copy()
    return d, anchors

In [ ]:


def strip_leading_zeros(folio_norm):
    """'001r' -> '1r', '012bisv' -> '12bisv' (ne touche pas au suffixe r/v/bis)."""
    if not folio_norm:
        return folio_norm
    m = re.match(r'^0*(\d+)(bis)?([rv])$', folio_norm)
    if m:
        return f"{m.group(1)}{m.group(2) or ''}{m.group(3)}"
    return folio_norm

In [131]:
def build_acte_report(df_grouped: pd.DataFrame, df_actes: pd.DataFrame, anchors: pd.DataFrame) -> pd.DataFrame:
    """
    ... (docstring inchangée) ...
    valid_folios : dict {register: set(folio_norm)} issu des images réelles
                   (load_images_list), utilisé pour vérifier la colonne FOLIO.
    """
    d = df_grouped[df_grouped["acte_seq_num"].notna()].copy()
    d["acte_seq_num"] = d["acte_seq_num"].astype(int)
    rows = []

    for register, g in d.groupby("register", sort=False):
        actes_vol = df_actes[df_actes["register"] == register].reset_index(drop=True)
        id_temp_to_idx = {v: i for i, v in enumerate(actes_vol["id_temp"])}

        mask = anchors["register"].map(register_num) == register_num(register)
        reg_anchors = anchors.loc[mask]

        offset_points = []
        for _, r in reg_anchors.iterrows():
            zone_id = str(r["zone_temp_id"])
            match = g[g["zone_temp_id"].astype(str) == zone_id]
            if match.empty:
                continue
            anchor_seq = int(match.iloc[0]["acte_seq_num"])
            target_idx = id_temp_to_idx.get(r["id_temp"])
            if target_idx is None:
                continue
            offset_points.append((anchor_seq, target_idx - (anchor_seq - 1)))
        offset_points.sort()

        def offset_for(seq_num, points=offset_points):
            applicable = [off for (s, off) in points if s <= seq_num]
            return applicable[-1] if applicable else 0

        for acte_seq_num, gg in g.sort_values("zone_order_on_folio").groupby("acte_seq_num"):
            start = gg[gg["acte_role"] == "start"]
            start = start.iloc[0] if len(start) else gg.iloc[0]

            offset = offset_for(acte_seq_num)
            ref_idx = acte_seq_num - 1 + offset
            has_ref = 0 <= ref_idx < len(actes_vol)

            warning = ""
            folio_raw = None
            if not has_ref:
                warning = "hors limites du tableau Acts CSV (avec offset appliqué)"
            elif "folio_norm" in actes_vol.columns:
                folio_raw = actes_vol.loc[ref_idx, "folio_raw"]
                folio_norm_acte = actes_vol.loc[ref_idx, "folio_norm"]
                
                folio_norm_zone = normalize_folio(start["folio_label"])
                if not folio_norm_acte:
                    warning = "FOLIO vide ou non reconnu dans Acts CSV"
                elif folio_norm_acte != folio_norm_zone:
                    if (register in FOLIO_LEADING_ZEROS_REGISTERS
                            and strip_leading_zeros(folio_norm_acte) == strip_leading_zeros(folio_norm_zone)):
                        pass  # écart uniquement dû au zéro-préfixage connu de ce registre
                    else:
                        warning = (f"FOLIO '{folio_raw}' (Acts CSV) ≠ folio réel de la zone "
                                   f"'{start['folio_label']}'")
                    
            gg_sorted = gg.assign(_ztid_num=gg["zone_temp_id"].astype(int)).sort_values("_ztid_num")

            rows.append({
                "register": register,
                "acte_seq_num": acte_seq_num,
                "offset_applique": offset,
                "id_temp_attendu": actes_vol.loc[ref_idx, "id_temp"] if has_ref else None,
                "act_number_positional": actes_vol.loc[ref_idx, "act_number"] if has_ref else None,
                "folio_label": start["folio_label"],
                "folio_acts_csv": folio_raw,
                "start_class": start["class_name"],
                "zone_temp_id_start": str(start["zone_temp_id"]),
                "zone_temp_ids": gg_sorted["zone_temp_id"].astype(str).tolist(),
                "zone_urls": gg_sorted["address_bvmm_path"].tolist(),
                "n_zones": len(gg),
                "iiif_url_page": start["urlImage"],
                "iiif_url_zone": start["address_bvmm_path"],
                "warning": warning,
            })

    return pd.DataFrame(rows)



## Template HTML

In [132]:
def export_acte_html(df_report: pd.DataFrame, df_orphans: pd.DataFrame, register: str, output_path: str):
    records = df_report[df_report["register"] == register].where(lambda x: x.notna(), None)
    orphans = df_orphans[df_orphans["register"] == register].where(lambda x: x.notna(), None)

    data = {
        "actes": records.to_dict("records"),
        "orphans": [
            {
                "zone_temp_id": str(o["zone_temp_id"]),
                "folio_label": o["folio_label"],
                "class_name": o["class_name"],
                "iiif_url_zone": o["address_bvmm_path"],   # <- idem, plus de reconstruction
            }
            for _, o in orphans.iterrows()
        ],
    }
    html = ACTE_HTML_TEMPLATE.replace("__REGISTER__", register).replace(
        "__DATA__", json.dumps(data, ensure_ascii=False))
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Rapport généré : {output_path}")


ACTE_HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="fr"><head><meta charset="UTF-8">
<title>Vérification des actes — __REGISTER__</title>
<style>
  body { font-family: -apple-system, "Segoe UI", Arial, sans-serif; margin: 24px;
         background:#f3f4f6; color:#222; }
  h1 { font-size: 1.4em; margin-bottom: 4px; }
  #summary { color:#444; margin-bottom: 16px; font-size: 14px; }

  table { border-collapse: separate; border-spacing: 0; width: 100%;
          background:#fff; border-radius: 8px; overflow: hidden;
          box-shadow: 0 1px 3px rgba(0,0,0,.08); }
  th, td { padding: 10px 12px; text-align: left; vertical-align: top;
           border-bottom: 1px solid #eee; font-size: 13px; }
  th { background: #2c3e50; color: #fff; position: sticky; top: 0;
       font-weight: 600; font-size: 12px; text-transform: uppercase;
       letter-spacing: .03em; }
  tr:hover td { background: #f7f9fb; }
  tr.mismatch td { background: #fbe4e6; }
  tr.mismatch:hover td { background: #f8d3d6; }
  tr.offset td { background: #eaf5ec; }
  tr.offset:hover td { background: #dcefe0; }

  .thumb-start img {
    max-width: 200px; max-height: 200px; width: auto; height: auto;
    display: block; border-radius: 6px; border: 1px solid #ddd;
    box-shadow: 0 1px 2px rgba(0,0,0,.15); cursor: pointer;
    transition: transform .12s ease;
  }
  .thumb-start img:hover { transform: scale(1.03); }

  .zonelist { display: flex; flex-wrap: wrap; gap: 8px; max-width: 640px; }
  .zonecard { text-align: center; }
  .zonecard img {
    max-width: 200px; max-height: 200px; width: auto; height: auto;
    display: block; border-radius: 6px; border: 1px solid #ddd;
    box-shadow: 0 1px 2px rgba(0,0,0,.15); cursor: pointer;
    transition: transform .12s ease;
  }
  .zonecard img:hover { transform: scale(1.03); }
  .zonecard .label { font-size: 10px; color: #888; margin-top: 2px; }

  .n-zones { font-size: 11px; color:#777; margin-top: 4px; }
  .warn-text { color:#a15c00; font-weight:600; }
  .badge { display:inline-block; padding: 1px 7px; border-radius: 10px;
           font-size: 11px; font-weight: 600; }
  .badge.offset { background:#dcefe0; color:#256029; }

  #orphans { margin-top: 32px; }
  #orphans h2 { font-size: 1.1em; }

  .lightbox { display:none; position:fixed; inset:0; background:rgba(0,0,0,.85);
              align-items:center; justify-content:center; z-index:10; }
  .lightbox img { max-height:90vh; max-width:90vw; border-radius: 6px; }
</style></head><body>

<h1>Vérification des actes — __REGISTER__</h1>
<div id="summary"></div>

<table><th>#</th><th>Offset</th><th>Act Number</th><th>id_temp attendu</th><th>Folio (zone)</th>
<th>Folio (Acts CSV)</th><th>Zones de l'acte</th><th>Avertissement</th>
<tbody id="tbody"></tbody></table>

<div id="orphans">
  <h2>Zones orphelines (AM/AF sans acte ouvert)</h2>
  <table><thead><tr><th>Folio</th><th>Classe</th><th>Zone</th></tr></thead>
  <tbody id="tbody-orphans"></tbody></table>
</div>

<div class="lightbox" id="lightbox" onclick="this.style.display='none'">
  <img id="lightbox-img" src="">
</div>

<script>
const data = __DATA__;

function showLightbox(url) {
  document.getElementById('lightbox-img').src = url.replace('/600,/', '/2200,/');
  document.getElementById('lightbox').style.display = 'flex';
}

const tbody = document.getElementById('tbody');
let warnCount = 0;
data.actes.forEach(r => {
  const tr = document.createElement('tr');
  if (r.warning) { tr.classList.add('mismatch'); warnCount++; }
  else if (r.offset_applique) { tr.classList.add('offset'); }

  const zonesHtml = r.zone_temp_ids.map((zid, i) =>
    `<div class="zonecard">
       <img src="${r.zone_urls[i]}" onclick="showLightbox('${r.zone_urls[i]}')">
       <div class="label">${zid}</div>
     </div>`
  ).join('');

  tr.innerHTML = `
    <td>${r.acte_seq_num}</td>
    <td>${r.id_temp_attendu ?? ''}</td>
    <td>${r.offset_applique ? `<span class="badge offset">${r.offset_applique}</span>` : ''}</td>
    <td>${r.act_number_positional ?? ''}</td>
    <td>${r.folio_label ?? ''}</td>
    <td>${r.folio_acts_csv ?? ''}</td>
    <td>
      <div class="zonelist">${zonesHtml}</div>
      <div class="n-zones">${r.n_zones} zone(s)</div>
    </td>
    <td class="warn-text">${r.warning ?? ''}</td>
  `;
  tbody.appendChild(tr);
});
document.getElementById('summary').textContent =
  `${data.actes.length} acte(s) — ${warnCount} avertissement(s)`;

const tbodyO = document.getElementById('tbody-orphans');
data.orphans.forEach(o => {
  const tr = document.createElement('tr');
  tr.innerHTML = `
    <td>${o.folio_label}</td>
    <td>${o.class_name}</td>
    <td>
      <div class="zonecard">
        <img src="${o.iiif_url_zone}" onclick="showLightbox('${o.iiif_url_zone}')">
        <div class="label">${o.zone_temp_id}</div>
      </div>
    </td>
  `;
  tbodyO.appendChild(tr);
});
</script>
</body></html>
"""

## Lancer vérification

In [133]:
df_actes = load_acts_list()
df_zones = load_zones_corrigees()

df_grouped = build_acte_groups(df_zones)
df_grouped, anchors = apply_acte_corrections(df_grouped)   # add_zone appliqué ici

df_report = build_acte_report(df_grouped, df_actes, anchors)  # set_start_zone recale ici
df_orphans = df_grouped[df_grouped["orphan"]]

for register in sorted(df_zones["register"].unique(), key=register_num):
    export_acte_html(df_report, df_orphans, register, f"verification_actes/verification_actes_{register}.html")

Rapport généré : verification_actes/verification_actes_JJ096.html
Rapport généré : verification_actes/verification_actes_JJ097.html
Rapport généré : verification_actes/verification_actes_JJ098.html
Rapport généré : verification_actes/verification_actes_JJ099.html
Rapport généré : verification_actes/verification_actes_JJ100.html
Rapport généré : verification_actes/verification_actes_JJ101.html
Rapport généré : verification_actes/verification_actes_JJ102.html
Rapport généré : verification_actes/verification_actes_JJ103.html
Rapport généré : verification_actes/verification_actes_JJ104.html
Rapport généré : verification_actes/verification_actes_JJ105.html
Rapport généré : verification_actes/verification_actes_JJ106.html
Rapport généré : verification_actes/verification_actes_JJ107.html
Rapport généré : verification_actes/verification_actes_JJ108.html
Rapport généré : verification_actes/verification_actes_JJ109.html
Rapport généré : verification_actes/verification_actes_JJ110.html
Rapport gé

## Overview erreurs potentielles

In [134]:
df_folio_mismatch = df_report[
    df_report["warning"].str.contains("FOLIO", na=False)
    & ~df_report["warning"].str.contains("hors limites", na=False)
].copy()

pd.set_option('display.max_rows', 500)

print(f"{len(df_folio_mismatch)} acte(s) avec décalage de folio")
df_folio_mismatch[["register", "acte_seq_num", "id_temp_attendu", "act_number_positional",
                    "folio_acts_csv", "folio_label", "warning"]].head(100)

7421 acte(s) avec décalage de folio


,register,acte_seq_num,id_temp_attendu,act_number_positional,folio_acts_csv,folio_label,warning
5975,JJ107,149,6051,148,67,67v,FOLIO '67' (Acts CSV) ≠ folio réel de la zone ...
9314,JJ116,3,9399,4,Vacat,6v,FOLIO vide ou non reconnu dans Acts CSV
10460,JJ119,150,10583,149,Vacat,98r,FOLIO vide ou non reconnu dans Acts CSV
11760,JJ122,252,11860,245,Vacat,123v,FOLIO vide ou non reconnu dans Acts CSV
11777,JJ122,269,11879,264,Vacat,132v,FOLIO vide ou non reconnu dans Acts CSV
12557,JJ125,1,12686,1,1,07r [=001r],FOLIO '1' (Acts CSV) ≠ folio réel de la zone '...
13489,JJ127,285,13613,285,177,176v,FOLIO '177' (Acts CSV) ≠ folio réel de la zone...
13784,JJ129,1,13934,1,1,Ir,FOLIO '1' (Acts CSV) ≠ folio réel de la zone 'Ir'
13785,JJ129,2,13935,2,1v,1r,FOLIO '1v' (Acts CSV) ≠ folio réel de la zone ...
13786,JJ129,3,13936,3,2,1v,FOLIO '2' (Acts CSV) ≠ folio réel de la zone '1v'


## Export CSV liaison acte-zones

In [135]:
rows = []
for _, r in df_report.iterrows():
    if r["id_temp_attendu"] is None:
        continue  # pas de correspondance possible (hors limites du tableau Acts CSV)
    for zid in r["zone_temp_ids"]:
        rows.append({
            "zone_temp_id": zid,
            "temporary_JJ96-211": r["id_temp_attendu"],
            "register": r["register"],
            "act_number": r["act_number_positional"],
            "warning": r["warning"],
        })

df_zone_to_act = pd.DataFrame(rows)

n_warn = (df_zone_to_act["warning"] != "").sum()
print(f"{len(df_zone_to_act)} zone(s) associée(s) à un acte")
print(f"dont {n_warn} avec un avertissement à vérifier avant import")

df_zone_to_act.to_csv("zone_to_act_heurist.csv", index=False, sep=";", encoding="utf-8-sig")

61873 zone(s) associée(s) à un acte
dont 14099 avec un avertissement à vérifier avant import
